<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Modèle d'appareil photo sténopé (Pinhole camera model)

Un appareil photo sténopé (pincole camera) est sans doute la représentation la plus simple du fonctionnement des appareils photo, y compris ceux utilisés sur le Duckiebot, qui produisent des images d'une scène en enregistrant les niveaux de lumière incidente réfléchie par les objets et atteignant le capteur. L'appareil photo sténopé idéal considère l'ouverture comme un point, mais la plupart des appareils combinent des ouvertures plus larges, laissant passer davantage de lumière, avec des lentilles qui focalisent la lumière réfléchie. En intégrant des modèles de distorsion de l'objectif, de nombreux appareils photo utilisés en pratique peuvent être modélisés comme des appareils photo sténopé.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/pinhole_camera_model/duckie-pinhole.png", width=500px>
  <p>Visualisation d'un simple appareil photo sténopé.</p>
  </div>
</figure>

Le modèle de la caméra sténopé est un modèle mathématique qui décrit la projection d'un point de l'espace tridimensionnel sur un plan image bidimensionnel par une caméra sténopé idéale, selon le principe de la projection perspective. Considérons un repère cartésien tridimensionnel dont l'origine est située au centre optique de la caméra (C) et dont l'axe Z positif est dirigé vers l'extérieur de la caméra. Cet axe Z est appelé axe principal. Le plan image bidimensionnel est perpendiculaire à l'axe principal et situé à une distance f derrière le centre de la caméra (soit z = -f), où f représente la distance focale. Afin d'éviter les images inversées, on peut considérer que le plan image est situé dans la direction Z positive, également à une distance f du centre de la caméra.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/pinhole_camera_model/pinhole-projection-a.png", width=500px>
  <p>géométrie de la caméra sténopé ().</p>
  </div>
</figure>

Considérons un point dans le repère de la caméra défini en termes de ses coordonnées cartésiennes $\mathbf{X}_\textrm{cam} = [X \; Y \; Z]^\top$. Nous pouvons exprimer ce point en *coordonnées homogènes* via le vecteur 4, $\tilde{\mathbf{X}}_\textrm{cam} = [X \; Y \; Z\; 1]^\top$. Le passage d'un point des coordonnées homogènes aux coordonnées cartésiennes implique de diviser par le dernier élément du vecteur de coordonnées homogènes (ici $1$) et de conserver tous les éléments sauf le dernier. La même règle s'applique sur le plan 2D de la caméra. Nous pouvons représenter un point $\mathbf{x}_\textrm{cam} = [X \; Y]$ sous sa forme homogène $\tilde{\mathbf{x}}_\textrm{cam} = [X \; Y \; 1]$.  
Ainsi, pour tout scalaire non nul $\alpha$, les coordonnées homogènes $[\alpha X \; \alpha Y \; \alpha]$ définissent les mêmes coordonnées cartésiennes $X, Y$ et représentent tous les points non homogènes dans le 3D qui correspondent au rayon passant par le point $[X \; Y]$ et le centre optimal de la caméra.

Nous pouvons relier le vecteur $\mathbf{X}_\textrm{cam}$ à sa projection sur l'image $\mathbf{x}$, exprimée en termes de son 3-vecteur homogène comme

$$ 
\begin{align}
\begin{bmatrix}
fx\\
fy\\
z
\end{bmatrix} &=
\begin{bmatrix}
f & 0 & 0 & 0\\
0 & f & 0 & 0\\
0 & 0 & 1 & 0
\end{bmatrix}
\begin{bmatrix}
x\\
y\\
z\\
1
\end{bmatrix}\\
\mathbf{x} &= P \mathbf{X}_\textrm{cam}
\end{align}
$$

where $P$ is the *camera projection matrix*.

Jusqu'à présent, nous avons supposé que l'origine du repère image se situait au point principal $\textbf{p}$, c'est-à-dire le point d'intersection de l'axe principal avec le plan image. En pratique, l'origine peut se trouver ailleurs et le point principal aura alors pour coordonnées $[p_x \; p_y]$. Par ailleurs, les appareils photo numériques peuvent comporter des pixels non carrés, ce qui est parfois représenté par des distances focales $(f_x, f_y)$ différentes dans les deux directions de l'espace image. L'ensemble de ces éléments conduit à une expression plus générale de la matrice de l'appareil photo :

$$ 
\mathbf{x} = 
\begin{bmatrix}
f_x x + p_x z\\
f_y y + p_y z\\
z
\end{bmatrix} =
\begin{bmatrix}
f_x & 0 & p_x & 0\\
0 & f_y & p_y & 0\\
0 & 0 & 1 & 0
\end{bmatrix}
\begin{bmatrix}
x\\
y\\
z\\
1
\end{bmatrix}
$$

We can express this as 

$$
\mathbf{x} = K [I \; \vert \; \mathbf{0}]\mathbf{X}_\textrm{cam} \qquad 
K = 
\begin{bmatrix}
f_x & s & p_x\\
0 & f_y & p_y\\
0 & 0 & 1
\end{bmatrix}
$$

où $K$ est la matrice d'étalonnage de la caméra, $I$ est une matrice identité $3 \times 3$ et $\mathbf{0}$ est un vecteur nul à trois éléments. 

**Remarque** : dans la matrice intrinsèque ci-dessus, nous avons introduit un paramètre d'asymétrie supplémentaire $s$ qui représente le cisaillement. Bien que l'asymétrie soit souvent (quasi-)nulle, nous l'inclurons dans la matrice intrinsèque pour la suite de l'exercice, par souci de généralité.

**Remarque** : K est la matrice que vous avez trouvée lors de la procédure d'étalonnage intrinsèque. Cependant, la caméra Duckiebot est plus complexe car elle possède un objectif qui doit également être modélisé. Par conséquent, après avoir effectué l'étalonnage intrinsèque, **il ne faut surtout pas modifier la mise au point de la caméra**, sous peine de devoir recommencer la procédure.

In [ ]:
### Exécutez cette cellule pour initialiser le problème

%matplotlib inline

import numpy as np
import matplotlib.pyplot as  plt
from mpl_toolkits.mplot3d import proj3d
from matplotlib.patches import FancyArrowPatch

class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        FancyArrowPatch.__init__(self, (0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        FancyArrowPatch.draw(self, renderer)

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        return np.min(zs)
        
class CameraProjection:
    def __init__(self, width, height, f):
        self._fig = plt.figure(figsize=(12,8), dpi= 100, facecolor='w', edgecolor='k')
        self._ax = self._fig.add_subplot(111, projection='3d')
        
        # Les coordonnées sont modifiées pour la visualisation.
        # x --> z, y --> x, z --> y
        self._ax.set_xlim(0, 4)
        self._ax.set_ylim(-(width/2 + 1), width/2 + 1)
        self._ax.set_zlim(-(height/2 + 1), height/2 + 1)
        self._ax.view_init(elev=15, azim=-75)
        self._width = width
        self._height = height
        self._f = f
        self._K = np.array([[f, 0, 0, 0],[0, f, 0, 0],[0, 0, 1, 0]])
        self._R = np.array([[0, 0, 1],[1, 0, 0], [0, 1, 0]])

    def transform(self, x):
        return self._R.dot(x)

    def transform(self, x, y, z):
        X = np.array([[x, y, z]]).transpose()
        return self._R.dot(X)

    def drawCameraModel(self):
        length = np.minimum(self._width, self._height)/2
        arrow_prop_dict = dict(mutation_scale=20, arrowstyle='->', shrinkA=0, shrinkB=0)
        a = Arrow3D([0, 0], [0, length], [0, 0], **arrow_prop_dict, color='r')
        self._ax.add_artist(a)
        a = Arrow3D([0, 0], [0, 0], [0, length], **arrow_prop_dict, color='g')
        self._ax.add_artist(a)
        a = Arrow3D([0, self._f*4], [0, 0], [0, 0], **arrow_prop_dict, color='b')
        self._ax.add_artist(a)
        
        # Image plane
        x = [self._width/2, self._width/2, -self._width/2, -self._width/2, self._width/2]
        y = [-self._height/2, self._height/2, self._height/2, -self._height/2, -self._height/2]
        z = [self._f, self._f, self._f, self._f, self._f]
        self._ax.plot(z, x, y)
        self._ax.plot([self._f], [0], [0], 'b.', markersize=12)

    def drawProjection(self, Xtilde):
        cam.drawCameraModel()
        X = np.vstack((Xtilde, 1))
        Xtransform =self._R.dot(X[0:3])
        x = self._K.dot(X)
        xtilde = np.vstack((x[0:2]/x[-1],self._f))
        xtransform = self._R.dot(xtilde)
        self._ax.set_xlim(0, Xtransform[0] + 2)
        ydim = np.array([width/2, np.abs(Xtilde[0])], dtype=object).max()
        self._ax.set_ylim(-ydim - 1, ydim + 1)
        zdim = np.array([height/2, np.abs(Xtilde[1])], dtype=object).max()
        self._ax.set_zlim(-zdim - 1, zdim + 1)
        
        self._ax.plot3D(xtransform[0], xtransform[1], xtransform[2],'r.', markersize=12)
        self._ax.plot3D(Xtransform[0], Xtransform[1], Xtransform[2],'r.', markersize=12)
        self._ax.plot3D([0, Xtransform[0][0]], [0,Xtransform[1][0]], [0,  Xtransform[2][0]],'k--')
        self._ax.plot3D(xtransform[0], xtransform[1], xtransform[2],'r.', markersize=12)
        self._ax.plot3D(Xtransform[0], Xtransform[1], Xtransform[2],'r.', markersize=12)
        plt.show()

### Exemple : Projection de la caméra

Dans cet exemple, vous allez expérimenter l'effet de la modification de la distance focale de la caméra. Nous supposerons que les distances focales selon les axes $x$ et $y$ sont identiques, c'est-à-dire $f_x = f_y = f$.

Le code suivant affichera la projection d'un point de la scène, dans le repère de la caméra, sur une image de dimensions $width$ × $height$, avec la distance focale spécifiée. Si vous augmentez la distance focale, que devient la distance entre le point projeté et le point principal ?

In [ ]:
width = 24                       # La largeur de l'image
height = 18                      # La hauteur de l'image
f = 4                            # La distance focale
Xtilde = np.array([[5, 5, 15]])  # Les coordonnées cartésiennes du point de la scène


cam = CameraProjection(width, height, f)
cam.drawProjection(Xtilde.transpose())

Dans de nombreux cas, nous nous intéressons au modèle qui décrit comment les points définis par rapport à un repère diffèrent de celui de la caméra (par exemple, un système de coordonnées du monde fixe ou le repère lié au corps du robot). Comme nous l'avons vu précédemment, nous pouvons transformer des points entre différents repères à l'aide de la matrice de rotation $3 \times 3$ $R$ et du vecteur de translation à trois éléments $\mathbf{t}$.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/pinhole_camera_model/pinhole-projection-b.png", width="600px" />
  <p>Les points exprimés dans un repère autre que celui de la caméra sont d'abord transformés dans le repère de la caméra avant d'être projetés sur l'image.</p>
  </div>
</figure>

Considérons un point de la scène $\mathbf{X}_w$ représenté par ses coordonnées non homogènes par rapport à un repère cartésien fixe. Soit $R$ la matrice de rotation du repère du monde vers le repère de la caméra, et soit $\mathbf{t}$ le vecteur de translation du repère du monde vers le repère de la caméra (c'est-à-dire l'origine du repère du monde exprimée dans le repère de la caméra). Nous pouvons exprimer les coordonnées cartésiennes du point de la scène par rapport au repère de la caméra comme suit :

$$\tilde{\mathbf{X}}_\textrm{cam} = R\tilde{\mathbf{X}}_w + \mathbf{t}$$

En termes de coordonnées homogènes, cela devient

$$\mathbf{X}_\textrm{cam} = 
\begin{bmatrix}
R\tilde{\mathbf{X}}_w + \mathbf{t}\\
1
\end{bmatrix}
$$

En reprenant l'expression de la matrice de la caméra ci-dessus, nous pouvons maintenant inclure la transformation du repère du monde vers le repère de la caméra dans notre modèle de caméra sténopé.

$$
\begin{align}
\mathbf{x} &= K [I \; \vert \; \mathbf{0}]\mathbf{X}_\textrm{cam}\\
           &= K [I \; \vert \; \mathbf{0}] \begin{bmatrix}
R\tilde{\mathbf{X}}_w + \mathbf{t}\\
1
\end{bmatrix}\\
&= K [R \; \vert \; \mathbf{t}] \mathbf{X}_w
\end{align}
$$

La première matrice, $K$, est appelée *matrice intrinsèque*, car elle contient des paramètres spécifiques (c'est-à-dire « intrinsèques ») à la caméra, indépendamment de sa position dans le monde. La seconde matrice, qui définit la transformation de coordonnées du repère du monde vers le repère de la caméra, est appelée *matrice extrinsèque*, car cette transformation ne dépend pas de la caméra (c'est-à-dire que les paramètres sont « externes » au choix de la caméra).

Vous pouvez maintenant passer au [notebook sur les homographies](../02-Homographies/homographies.ipynb).